# JMQ 수익성 팩터 기반 투자 종목 선정 파이프라인

**팩터**: `profitability_v3` — GPOA, CFOA, GMAR, ROA, ROE 5개 수익성 지표의 횡단면 순위 Z-스코어 평균  
**유니버스**: KOSPI + KOSDAQ 제조업  
**리밸런싱**: 분기 (QE)

## 종목 선정 순서

1. 전체 유니버스에서 **60일 평균 거래대금 하위 10% 제외**
2. 시가총액 5분위 → **1분위(소형주)** 선택
3. 팩터 5분위 → **1분위(수익성 상위)** 선택 *(팩터값 낮을수록 수익성 좋음)*

## 필요한 입력 파일

`../../00_input/` 경로 (전기간 데이터, 업데이트 불필요):
```
# 팩터 계산용 (재무 데이터)
KOSPI_KOSDAQ_제조업_gross_profit.csv     # 매출총이익 (연결)
KOSPI_KOSDAQ_제조업_cash_flow.csv        # 영업현금흐름 (연결)
KOSPI_KOSDAQ_제조업_sales.csv            # 매출액 (연결)
KOSPI_KOSDAQ_제조업_total_asset.csv      # 총자산 (연결)
KOSPI_KOSDAQ_제조업_ROE.csv              # 자기자본수익률 (분기)
KOSPI_KOSDAQ_제조업_ROA.csv              # 자산수익률 (분기)

# 종목 선정 및 포트폴리오 구성용
KOSPI_KOSDAQ_제조업_close.csv                   # 실제 종가 (매수가 계산용)
KOSPI_KOSDAQ_제조업_mkt_cap.csv          # 시가총액
KOSPI_KOSDAQ_제조업_trading_value_60.csv # 60일 거래대금 평균
```

`input/` 경로 (선택, 섹터 정보):
```
KOSPI_KOSDAQ_sector (날짜).csv           # 섹터 정보 (없으면 자동 생략)
```

## 출력 파일

```
investment_portfolio_JMQ_profitability_YYYYMMDD.xlsx
```

## 사용 방법

### 1. 설정 수정
- **Cell 3** "설정 파라미터"에서 수정
  - `TARGET_DATE`: 투자 기준일 (분기말, e.g. `'2026-03-31'`)
  - `INVESTMENT_AMOUNT`: 투자 금액
  - `PREVIOUS_QUARTER_FILE`: 저번 분기 포트폴리오 엑셀 경로 (리밸런싱 비교용)

### 2. 노트북 실행
- 상단 메뉴 `Cell > Run All` 또는 `Shift + Enter`로 실행
- 전체 실행 시간: **1분 이내**

### 3. 결과 확인
- 엑셀 파일 시트:
  - **포트폴리오**: 선택된 종목 상세 (시가총액 내림차순)
  - **섹터별통계**: 섹터별 투자 비중
  - **요약**: 포트폴리오 요약
  - **계속보유_변화없음** / **계속보유_주수조정** / **매도종목** / **신규매수종목** / **비교요약** (리밸런싱 비교 시)



---
# 필요한 데이터 (Dataguide 기준, `../../00_input/` 에 저장)

**팩터 계산용 재무 데이터 (연결 기준, 분기말 인덱스)**

| 파일명 | 내용 | 래그 |
|--------|------|------|
| `KOSPI_KOSDAQ_제조업_gross_profit.csv` | 매출총이익 | 14 거래일 |
| `KOSPI_KOSDAQ_제조업_cash_flow.csv` | 영업현금흐름 | 14 거래일 |
| `KOSPI_KOSDAQ_제조업_sales.csv` | 매출액 | 14 거래일 |
| `KOSPI_KOSDAQ_제조업_total_asset.csv` | 총자산 | 14 거래일 |
| `KOSPI_KOSDAQ_제조업_ROE.csv` | 자기자본수익률 (분기) | 5 거래일 |
| `KOSPI_KOSDAQ_제조업_ROA.csv` | 자산수익률 (분기) | 5 거래일 |

**종목 선정 및 포트폴리오 구성용**

| 파일명 | 내용 |
|--------|------|
| `KOSPI_KOSDAQ_제조업_close.csv` | 실제 종가 (매수가 계산용) |
| `KOSPI_KOSDAQ_제조업_mkt_cap.csv` | 시가총액 (분기말 인덱스) |
| `KOSPI_KOSDAQ_제조업_trading_value_60.csv` | 60일 평균 거래대금 (분기말 인덱스) |

**선택 데이터 (섹터 정보, `input/` 에 저장)**

| 파일명 | 내용 |
|--------|------|
| `KOSPI_KOSDAQ_sector (날짜).csv` | 거래소 업종 정보 (없어도 실행 가능) |

---
# 설정 파라미터

아래 설정값들을 수정하여 원하는 조건으로 투자 종목을 선정할 수 있습니다.


In [1]:
# ==================== 전역 설정 파라미터 ====================

# 1. 투자 기준일 (분기말 날짜, e.g. '2026-03-31')
TARGET_DATE = '2026-03-31'  # None으로 설정하면 팩터 데이터의 가장 최근 날짜 사용

# 2. 거래대금 필터 (전체 유니버스 기준 하위 N% 제외)
TRADING_THRESHOLD = 0.1    # 0.1 = 하위 10% 제외

# 3. 시가총액 분위 설정
NUM_MKTCAP_QUANTILES = 5   # 몇 분위로 나눌지
MKTCAP_QUANTILE_IDX  = 1   # 선택할 분위 인덱스 (1=최소시총, NUM_MKTCAP_QUANTILES=최대시총)
                            # 1 = 소형주 하위 20%

# 4. 팩터 분위 설정
NUM_FACTOR_QUANTILES = 5   # 몇 분위로 나눌지
FACTOR_QUANTILE_IDX  = 1   # 선택할 분위 인덱스 (1=팩터값 낮음=수익성 높음)

# 5. 투자 금액 설정
INVESTMENT_AMOUNT = 10000000  # 투자 금액 (원)

# 6. 필터링 옵션
EXCLUDE_PENNY_STOCK = True   # 동전주 제외 여부
PENNY_STOCK_PRICE   = 1000   # 동전주 기준 (원)

# 7. 상장폐지/거래정지 종목 제외 (수동 리스트)
DELISTED_STOCKS = [
    # 예: '종목명1', '종목명2'
]

# 8. 입력 파일 경로
INPUT_PATH = '../../00_input/'

# 9. 리밸런싱 비교 옵션
ENABLE_REBALANCING_COMPARISON = False  # False로 설정하면 비교 기능 비활성화
PREVIOUS_QUARTER_FILE = None           # 저번 분기 포트폴리오 엑셀 파일 경로
                                       # 예: "Raw/investment_portfolio_JMQ_20251231.xlsx"

print("=" * 60)
print("설정 완료")
print("=" * 60)
print(f"팩터:          profitability_v3  (노트북 내 직접 계산)")
print(f"투자 기준일:   {TARGET_DATE}")
print(f"거래대금 필터: 전체 유니버스 기준 하위 {TRADING_THRESHOLD*100:.0f}% 제외")
print(f"시가총액:      {NUM_MKTCAP_QUANTILES}분위 중 {MKTCAP_QUANTILE_IDX}분위 (소형주)")
print(f"팩터 분위:     {NUM_FACTOR_QUANTILES}분위 중 {FACTOR_QUANTILE_IDX}분위 (수익성 상위)")
print(f"투자 금액:     {INVESTMENT_AMOUNT:,}원")
if ENABLE_REBALANCING_COMPARISON and PREVIOUS_QUARTER_FILE:
    print(f"리밸런싱 비교: 활성화 (저번 분기 파일: {PREVIOUS_QUARTER_FILE})")
elif ENABLE_REBALANCING_COMPARISON:
    print("리밸런싱 비교: 활성화 (저번 분기 파일 미설정)")
else:
    print("리밸런싱 비교: 비활성화")
print("=" * 60)

설정 완료
팩터:          profitability_v3  (노트북 내 직접 계산)
투자 기준일:   2026-03-31
거래대금 필터: 전체 유니버스 기준 하위 10% 제외
시가총액:      5분위 중 1분위 (소형주)
팩터 분위:     5분위 중 1분위 (수익성 상위)
투자 금액:     10,000,000원
리밸런싱 비교: 비활성화


---
# Step 1: 데이터 로드 및 팩터 계산

재무 데이터를 불러와 `profitability_v3` 팩터를 직접 계산하고, 종목 선정에 필요한 보조 데이터도 로드합니다.

**팩터 계산 방법 (QMJ 수익성 팩터)**
- 각 지표를 횡단면 내림차순 순위로 변환 → 횡단면 Z-스코어 → 5개 Z-스코어 평균
- 팩터값 **낮을수록 수익성 좋은 종목**


In [2]:
import pandas as pd
import numpy as np
import glob
import os

print("Step 1: 데이터 로드 및 팩터 계산 시작")
print("=" * 60)

# ── [1/3] 재무 데이터 로드 (팩터 계산용) ─────────────────────────────────
print("[1/3] 재무 데이터 로드 중...")

# 연결 재무 데이터
gross_profit = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_gross_profit (26-03-31).csv', index_col='Date', parse_dates=True)
cashflow     = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_영업활동으로인한현금흐름 (26-03-31).csv',    index_col='Date', parse_dates=True)
sales        = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_sales (26-03-31).csv',        index_col='Date', parse_dates=True)
total_assets = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_total_asset (26-03-31).csv',  index_col='Date', parse_dates=True)

# 분기 재무 데이터
ROE = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_ROE (26-03-31).csv', index_col='Date', parse_dates=True)
ROA = pd.read_csv(INPUT_PATH + 'KOSPI_KOSDAQ_제조업_ROA (26-03-31).csv', index_col='Date', parse_dates=True)

print(f"  ✓ gross_profit: {gross_profit.shape}")
print(f"  ✓ cashflow:     {cashflow.shape}")
print(f"  ✓ sales:        {sales.shape}")
print(f"  ✓ total_assets: {total_assets.shape}")
print(f"  ✓ ROE:          {ROE.shape}")
print(f"  ✓ ROA:          {ROA.shape}")

Step 1: 데이터 로드 및 팩터 계산 시작
[1/3] 재무 데이터 로드 중...
  ✓ gross_profit: (1, 2482)
  ✓ cashflow:     (1, 2482)
  ✓ sales:        (1, 2482)
  ✓ total_assets: (1, 2482)
  ✓ ROE:          (1, 2482)
  ✓ ROA:          (1, 2482)


In [3]:
# 기준 날짜에서 각 재무 데이터의 유효 종목 수 확인
check_date = gross_profit.index.max()
print(f"\n[유효값 확인] 기준일: {check_date.date()}")
print("-" * 50)

fin_data = {
    'gross_profit': gross_profit,
    'cashflow    ': cashflow,
    'sales       ': sales,
    'total_assets': total_assets,
    'ROE         ': ROE,
    'ROA         ': ROA,
}

valid_index = None
for name, df in fin_data.items():
    row = df.loc[check_date] if check_date in df.index else pd.Series(dtype=float)
    valid = row.dropna().index
    print(f"  {name}: {len(valid):,}개")
    valid_index = valid if valid_index is None else valid_index.intersection(valid)

print(f"\n  ★ 모든 재무 데이터 유효한 공통 종목: {len(valid_index):,}개")

# 공통 유효 종목만 남기도록 모든 재무 데이터 필터링
gross_profit = gross_profit[valid_index]
cashflow     = cashflow[valid_index]
sales        = sales[valid_index]
total_assets = total_assets[valid_index]
ROE          = ROE[valid_index]
ROA          = ROA[valid_index]

print(f"\n  → 재무 데이터 필터링 완료: 모든 지표 {len(valid_index):,}개 종목 기준으로 통일")


[유효값 확인] 기준일: 2025-12-31
--------------------------------------------------
  gross_profit: 1,732개
  cashflow    : 1,734개
  sales       : 2,338개
  total_assets: 1,734개
  ROE         : 1,695개
  ROA         : 1,695개

  ★ 모든 재무 데이터 유효한 공통 종목: 1,668개

  → 재무 데이터 필터링 완료: 모든 지표 1,668개 종목 기준으로 통일


In [4]:

# ── [2/3] profitability_v3 팩터 계산 ─────────────────────────────────────
print("\n[2/3] profitability_v3 팩터 계산 중...")

# 지표 계산
GPOA = gross_profit / total_assets   # 총이익 / 총자산
CFOA = cashflow     / total_assets   # 영업현금흐름 / 총자산
GMAR = gross_profit / sales          # 매출총이익률

# 횡단면 내림차순 순위 (값이 높을수록 좋음 → 낮은 순위가 좋은 종목)
def rank_desc(df):
    return df.rank(axis=1, ascending=False, method='first', na_option='keep').astype('Int64')

GPOA_rank = rank_desc(GPOA)
CFOA_rank = rank_desc(CFOA)
GMAR_rank = rank_desc(GMAR)
ROA_rank  = rank_desc(ROA)
ROE_rank  = rank_desc(ROE)

# 횡단면 Z-스코어 (모집단 표준편차 기준)
def zscore_by_date(rank_df):
    r        = rank_df.astype(float)
    mean     = r.mean(axis=1, skipna=True)
    std      = r.std(axis=1,  skipna=True, ddof=0)
    std_safe = std.where(std != 0)
    return r.sub(mean, axis=0).div(std_safe, axis=0)

GPOA_z = zscore_by_date(GPOA_rank)
CFOA_z = zscore_by_date(CFOA_rank)
GMAR_z = zscore_by_date(GMAR_rank)
ROA_z  = zscore_by_date(ROA_rank)
ROE_z  = zscore_by_date(ROE_rank)

# 5개 Z-스코어 평균 → profitability_v3
# ※ 값이 낮을수록 수익성 좋은 종목
factor_df = (
    pd.concat([GPOA_z, CFOA_z, GMAR_z, ROA_z, ROE_z],
              keys=['GPOA', 'CFOA', 'GMAR', 'ROA', 'ROE'])
    .groupby(level=1)
    .mean()
)

print(f"  ✓ 팩터 계산 완료: {factor_df.shape}")
print(f"    기간: {factor_df.index.min().date()} ~ {factor_df.index.max().date()}")

# ── [3/3] 보조 데이터 로드 ───────────────────────────────────────────────
print("\n[3/3] 보조 데이터 로드 중...")

mktcap_df = pd.read_csv(
    INPUT_PATH + 'KOSPI_KOSDAQ_제조업_mkt_cap.csv', index_col='Date', parse_dates=True
)
print(f"  ✓ 시가총액: {mktcap_df.shape}")

trading_value_df = pd.read_csv(
    INPUT_PATH + 'KOSPI_KOSDAQ_제조업_trading_value_60.csv', index_col='Date', parse_dates=True
)
print(f"  ✓ 60일 거래대금: {trading_value_df.shape}")

# 실제 종가 (매수가 계산용 — 수정종가 아닌 실제 거래 가격)
close_df = pd.read_csv(
    INPUT_PATH + 'KOSPI_KOSDAQ_close.csv', index_col='Date', parse_dates=True
)
print(f"  ✓ 실제 종가: {close_df.shape}")

# 섹터 데이터 (선택적 — 없어도 실행 가능)
try:
    sector_files = sorted(glob.glob('input/KOSPI_KOSDAQ_sector*.csv'))
    if sector_files:
        sector_df = pd.read_csv(sector_files[-1], index_col=0)
        sector_df = sector_df.T
        print(f"  ✓ 섹터: {sector_df.shape}  ({os.path.basename(sector_files[-1])})")
    else:
        sector_df = None
        print("  - 섹터 파일 없음: 섹터 정보 생략")
except Exception:
    sector_df = None
    print("  - 섹터 데이터 로드 실패: 섹터 정보 생략")

print("\n" + "=" * 60)
print("Step 1 완료!")
print("=" * 60)


[2/3] profitability_v3 팩터 계산 중...
  ✓ 팩터 계산 완료: (1, 1668)
    기간: 2025-12-31 ~ 2025-12-31

[3/3] 보조 데이터 로드 중...
  ✓ 시가총액: (436, 3503)
  ✓ 60일 거래대금: (436, 3503)
  ✓ 실제 종가: (9345, 3948)
  - 섹터 파일 없음: 섹터 정보 생략

Step 1 완료!


---
# Step 2: 종목 선정

JMQ 수익성 팩터(profitability_v3) 기반으로 3단계 필터를 적용합니다.

1. 전체 유니버스 기준 거래대금 하위 10% 제외
2. 시가총액 5분위 → 1분위(소형주) 선택
3. 팩터 5분위 → 1분위(수익성 상위) 선택


In [5]:
print("\n" + "=" * 60)
print("Step 2: 종목 선정 시작")
print("=" * 60)

# 투자 기준일 결정
if TARGET_DATE is None:
    target_date = factor_df.index.max()
    print(f"투자 기준일: {target_date.strftime('%Y-%m-%d')} (팩터 데이터 최신 날짜)")
else:
    target_date = pd.to_datetime(TARGET_DATE)
    print(f"투자 기준일: {target_date.strftime('%Y-%m-%d')}")

# 팩터 데이터에서 가장 가까운 날짜 선택
if target_date not in factor_df.index:
    available = factor_df.index[factor_df.index <= target_date]
    if len(available) == 0:
        raise ValueError(f"팩터 데이터에 {target_date.date()} 이전 날짜가 없습니다.")
    target_date = available.max()
    print(f"  → 가장 가까운 팩터 날짜 사용: {target_date.strftime('%Y-%m-%d')}")

# ── Step A: 거래대금 하위 10% 제외 (전체 유니버스 기준) ──────────────────
print(f"\n[Step A] 거래대금 필터 (전체 유니버스 기준 하위 {TRADING_THRESHOLD*100:.0f}% 제외)")

if target_date not in trading_value_df.index:
    tv_dates = trading_value_df.index[trading_value_df.index <= target_date]
    tv_date  = tv_dates.max() if len(tv_dates) > 0 else target_date
else:
    tv_date = target_date

trading_series  = trading_value_df.loc[tv_date].dropna()
tv_threshold    = trading_series.quantile(TRADING_THRESHOLD)
liquid_tickers  = trading_series[trading_series > tv_threshold].index
print(f"  전체 종목 수: {len(trading_series):,}")
print(f"  거래대금 임계값: {tv_threshold:,.0f}원")
print(f"  필터링 후: {len(liquid_tickers):,}개")

# ── Step B: 시가총액 N분위 → 선택 분위 필터 ──────────────────────────────
print(f"\n[Step B] 시가총액 {NUM_MKTCAP_QUANTILES}분위 중 {MKTCAP_QUANTILE_IDX}분위 선택 (소형주)")

if target_date not in mktcap_df.index:
    mc_dates = mktcap_df.index[mktcap_df.index <= target_date]
    mc_date  = mc_dates.max() if len(mc_dates) > 0 else target_date
else:
    mc_date = target_date

mktcap_series   = mktcap_df.loc[mc_date, liquid_tickers].dropna()
cap_labels      = pd.qcut(mktcap_series, q=NUM_MKTCAP_QUANTILES, labels=False)
mktcap_filtered = mktcap_series[cap_labels == (MKTCAP_QUANTILE_IDX - 1)].index
print(f"  거래대금 필터 후 종목 수: {len(mktcap_series):,}")
print(f"  시가총액 {MKTCAP_QUANTILE_IDX}분위 종목 수: {len(mktcap_filtered):,}")

# ── Step C: 팩터 N분위 → 선택 분위 선택 ─────────────────────────────────
print(f"\n[Step C] 팩터 {NUM_FACTOR_QUANTILES}분위 중 {FACTOR_QUANTILE_IDX}분위 선택 (수익성 상위)")

# 수정 코드
factor_series = factor_df.loc[target_date, mktcap_filtered.intersection(factor_df.columns)].dropna()
factor_labels   = pd.qcut(factor_series, q=NUM_FACTOR_QUANTILES, labels=False, duplicates='drop')
selected_tickers = factor_series[factor_labels == (FACTOR_QUANTILE_IDX - 1)].index
selected_factor  = factor_series.loc[selected_tickers]
print(f"  시가총액 필터 후 팩터 있는 종목: {len(factor_series):,}")
print(f"  팩터 {FACTOR_QUANTILE_IDX}분위 선택 종목 수: {len(selected_tickers):,}")

# 상장폐지 종목 제외
if DELISTED_STOCKS:
    delisted_in_selected = [s for s in DELISTED_STOCKS if s in selected_tickers]
    if delisted_in_selected:
        print(f"\n  상장폐지 종목 제외: {delisted_in_selected}")
        selected_tickers = selected_tickers.drop(delisted_in_selected)
        selected_factor  = selected_factor.loc[selected_tickers]

print(f"\n최종 선택 종목 수: {len(selected_tickers):,}")


Step 2: 종목 선정 시작
투자 기준일: 2026-03-31
  → 가장 가까운 팩터 날짜 사용: 2025-12-31

[Step A] 거래대금 필터 (전체 유니버스 기준 하위 10% 제외)
  전체 종목 수: 2,494
  거래대금 임계값: 77,917,298원
  필터링 후: 2,244개

[Step B] 시가총액 5분위 중 1분위 선택 (소형주)
  거래대금 필터 후 종목 수: 2,244
  시가총액 1분위 종목 수: 449

[Step C] 팩터 5분위 중 1분위 선택 (수익성 상위)
  시가총액 필터 후 팩터 있는 종목: 287
  팩터 1분위 선택 종목 수: 58

최종 선택 종목 수: 58


---
# Step 3: 포트폴리오 구성

선택된 종목의 종가를 조회하고, 등가중 포트폴리오와 매수 수량을 계산합니다.


In [6]:
print("\n" + "=" * 60)
print("Step 3: 포트폴리오 구성 시작")
print("=" * 60)

# 종가 (매수가) 데이터에서 기준일 추출
print(f"\n종가 데이터 추출 중...")
if target_date not in close_df.index:
    cl_dates = close_df.index[close_df.index <= target_date]
    cl_date  = cl_dates.max() if len(cl_dates) > 0 else target_date
else:
    cl_date = target_date

close_at_date = close_df.loc[cl_date]

# 선택된 종목 종가 가져오기
available_prices = close_at_date.reindex(selected_tickers).dropna()
print(f"✓ 종가 데이터가 있는 종목 수: {len(available_prices)}")

if len(available_prices) < len(selected_tickers):
    missing = len(selected_tickers) - len(available_prices)
    print(f"  - 경고: {missing}개 종목의 종가 데이터가 없습니다.")
    selected_tickers = available_prices.index
    selected_factor  = selected_factor.loc[selected_tickers]
    available_prices = available_prices.loc[selected_tickers]

# 동전주 제외
if EXCLUDE_PENNY_STOCK:
    penny = available_prices[available_prices <= PENNY_STOCK_PRICE].index
    if len(penny) > 0:
        print(f"✓ 동전주(종가 {PENNY_STOCK_PRICE}원 이하) 제외: {len(penny)}개")
        selected_tickers = available_prices[available_prices > PENNY_STOCK_PRICE].index
        selected_factor  = selected_factor.loc[selected_tickers]
        available_prices = available_prices.loc[selected_tickers]

print(f"\n최종 투자 대상 종목 수: {len(selected_tickers)}")


Step 3: 포트폴리오 구성 시작

종가 데이터 추출 중...
✓ 종가 데이터가 있는 종목 수: 58
✓ 동전주(종가 1000원 이하) 제외: 5개

최종 투자 대상 종목 수: 53


In [7]:
# 등가중 포트폴리오 구성
num_selected = len(selected_tickers)
equal_weight = 1.0 / num_selected

weight_df = pd.DataFrame({
    '종목코드': selected_tickers,
    '팩터값':   selected_factor.loc[selected_tickers].values,
    '종가':     available_prices.loc[selected_tickers].values,
    '가중치':   equal_weight
})

weight_df['투자금액']    = weight_df['가중치'] * INVESTMENT_AMOUNT
weight_df['매수개수']    = (weight_df['투자금액'] / weight_df['종가']).round().astype(int)
weight_df['실제투자금액'] = weight_df['매수개수'] * weight_df['종가']

# 시가총액 추가
mktcap_at_date       = mktcap_df.loc[mc_date, selected_tickers]
weight_df['시가총액'] = mktcap_at_date.values

# 섹터 추가
if sector_df is not None:
    sector_column = sector_df.columns[0]
    def clean_sector(ticker):
        if ticker in sector_df.index:
            val = sector_df.loc[ticker, sector_column]
            if isinstance(val, str):
                val = val.replace('코스피 ', '').replace('코스닥 ', '')
            return val
        return '정보없음'
    weight_df['섹터'] = weight_df['종목코드'].map(clean_sector)
else:
    weight_df['섹터'] = '제조업'

weight_df = weight_df.sort_values('시가총액', ascending=False).reset_index(drop=True)
total_investment = weight_df['실제투자금액'].sum()

print("\n" + "=" * 60)
print("포트폴리오 구성 결과")
print("=" * 60)
print(f"투자 기준일:    {target_date.strftime('%Y-%m-%d')}")
print(f"선택된 종목 수: {num_selected}")
print(f"목표 투자 금액: {INVESTMENT_AMOUNT:,}원")
print(f"실제 투자 금액: {total_investment:,.0f}원")
print(f"투자 비율:      {total_investment/INVESTMENT_AMOUNT*100:.2f}%")
print("=" * 60)


포트폴리오 구성 결과
투자 기준일:    2025-12-31
선택된 종목 수: 53
목표 투자 금액: 10,000,000원
실제 투자 금액: 9,989,781원
투자 비율:      99.90%


In [8]:
# 결과 출력
print("\n투자 포트폴리오 상세 (시가총액 기준 내림차순):")
print(weight_df.head(20).to_string(index=False))
print(f"... (총 {len(weight_df)}개 종목)")

print("\n\n포트폴리오 통계:")
print(f"평균 팩터값:     {weight_df['팩터값'].mean():.6f}")
print(f"팩터값 표준편차: {weight_df['팩터값'].std():.6f}")
print(f"최소 팩터값:     {weight_df['팩터값'].min():.6f}")
print(f"최대 팩터값:     {weight_df['팩터값'].max():.6f}")
print(f"평균 종가:       {weight_df['종가'].mean():,.0f}원")
print(f"평균 투자금액:   {weight_df['실제투자금액'].mean():,.0f}원")
print(f"평균 시가총액:   {weight_df['시가총액'].mean():,.0f}원")

# 섹터별 비중 통계
sector_stats = weight_df.groupby('섹터').agg(
    실제투자금액=('실제투자금액', 'sum'),
    종목수=('종목코드', 'count')
)
sector_stats['비중(%)'] = (sector_stats['실제투자금액'] / total_investment * 100).round(2)
sector_stats = sector_stats.sort_values('비중(%)', ascending=False)
print("\n\n섹터별 비중 통계:")
print(sector_stats.to_string())


투자 포트폴리오 상세 (시가총액 기준 내림차순):
    종목코드       팩터값     종가      가중치          투자금액  매수개수   실제투자금액    시가총액  섹터
   티디에스팜 -0.658553 9850.0 0.018868 188679.245283    19 187150.0 54471.0 제조업
     이글벳 -0.819298 4185.0 0.018868 188679.245283    45 188325.0 52906.0 제조업
   일월지엠엘 -1.153662 2800.0 0.018868 188679.245283    67 187600.0 52190.0 제조업
     엔텔스 -0.680983 5080.0 0.018868 188679.245283    37 187960.0 52044.0 제조업
  제이에스티나 -0.492409 3140.0 0.018868 188679.245283    60 188400.0 51822.0 제조업
      파수 -1.004548 4415.0 0.018868 188679.245283    43 189845.0 51673.0 제조업
   모비데이즈 -0.837989 1585.0 0.018868 188679.245283   119 188615.0 50980.0 제조업
제이투케이바이오 -0.855849 8600.0 0.018868 188679.245283    22 189200.0 50288.0 제조업
 영림원소프트랩 -0.997487 6140.0 0.018868 188679.245283    31 190340.0 49924.0 제조업
    프로이천 -0.900708 1724.0 0.018868 188679.245283   109 187916.0 48603.0 제조업
    웨이버스 -0.841727 1009.0 0.018868 188679.245283   187 188683.0 48589.0 제조업
    케이씨티 -0.464580 2795.0 0.018868 188679.245283    68 1900

In [9]:
# 결과를 XLSX 파일로 저장
output_filename = f"investment_portfolio_JMQ_profitability_{target_date.strftime('%Y%m%d')}.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    weight_df.to_excel(writer, sheet_name='포트폴리오', index=False)
    sector_stats.to_excel(writer, sheet_name='섹터별통계')

    summary = {
        '투자기준일':    target_date.strftime('%Y-%m-%d'),
        '팩터':          'profitability_v3',
        '거래대금_필터': f'전체 유니버스 기준 하위 {TRADING_THRESHOLD*100:.0f}% 제외',
        '시가총액_분위': f'{NUM_MKTCAP_QUANTILES}분위 중 {MKTCAP_QUANTILE_IDX}분위 (소형주)',
        '팩터_분위':     f'{NUM_FACTOR_QUANTILES}분위 중 {FACTOR_QUANTILE_IDX}분위 (수익성 상위)',
        '종목수':        num_selected,
        '목표투자금액':   INVESTMENT_AMOUNT,
        '실제투자금액':   total_investment,
        '평균팩터값':     weight_df['팩터값'].mean(),
        '최소팩터값':     weight_df['팩터값'].min(),
        '최대팩터값':     weight_df['팩터값'].max(),
        '평균시가총액':   weight_df['시가총액'].mean()
    }
    pd.DataFrame([summary]).to_excel(writer, sheet_name='요약', index=False)

print(f"\n✓ 결과가 '{output_filename}' 파일로 저장되었습니다.")
print(f"  - 시트1: 포트폴리오 (시가총액 기준 내림차순 정렬)")
print(f"  - 시트2: 섹터별통계")
print(f"  - 시트3: 요약")
print("\n" + "=" * 60)
print("Step 3 완료!")
print("=" * 60)
print("\n" + "=" * 60)
print("모든 단계 완료!")
print("=" * 60)
print(f"\n최종 결과 파일: {output_filename}")
print(f"총 {num_selected}개 종목, 실제 투자 금액 {total_investment:,.0f}원")


✓ 결과가 'investment_portfolio_JMQ_profitability_20251231.xlsx' 파일로 저장되었습니다.
  - 시트1: 포트폴리오 (시가총액 기준 내림차순 정렬)
  - 시트2: 섹터별통계
  - 시트3: 요약

Step 3 완료!

모든 단계 완료!

최종 결과 파일: investment_portfolio_JMQ_profitability_20251231.xlsx
총 53개 종목, 실제 투자 금액 9,989,781원


---
# Step 4: 리밸런싱 비교 (선택사항)

저번 달 포트폴리오와 비교하여 계속 보유, 주수 조정, 매도, 신규 매수 종목을 구분합니다.


In [10]:
# ==================== 리밸런싱 비교 (선택사항) ====================
if ENABLE_REBALANCING_COMPARISON and PREVIOUS_QUARTER_FILE:
    print("\n" + "=" * 60)
    print("저번 달 리밸런싱 데이터와 비교")
    print("=" * 60)
    
    try:
        # 저번 분기 포트폴리오 데이터 불러오기
        previous_portfolio = pd.read_excel(PREVIOUS_QUARTER_FILE, sheet_name='포트폴리오')
        print(f"\n저번 분기 포트폴리오 파일 로드 완료: {PREVIOUS_QUARTER_FILE}")
        print(f"저번 분기 종목 수: {len(previous_portfolio)}")
        
        # 이번 달 포트폴리오 (weight_df)
        current_portfolio = weight_df.copy()
        print(f"이번 분기 종목 수: {len(current_portfolio)}")
        
        # 종목코드 컬럼명 확인 및 통일
        prev_ticker_col = '종목코드' if '종목코드' in previous_portfolio.columns else previous_portfolio.columns[0]
        prev_shares_col = '매수개수' if '매수개수' in previous_portfolio.columns else '주수' if '주수' in previous_portfolio.columns else None
        prev_price_col = '종가' if '종가' in previous_portfolio.columns else None
        
        if prev_shares_col is None:
            # 매수개수 컬럼이 없으면 찾기
            for col in previous_portfolio.columns:
                if '개수' in str(col) or '주수' in str(col) or 'shares' in str(col).lower():
                    prev_shares_col = col
                    break
        
        if prev_shares_col is None:
            raise ValueError("저번 달 포트폴리오에서 매수개수/주수 컬럼을 찾을 수 없습니다.")
        
        # 종가 컬럼 찾기
        if prev_price_col is None:
            for col in previous_portfolio.columns:
                if '종가' in str(col) or '가격' in str(col) or 'price' in str(col).lower() or 'close' in str(col).lower():
                    prev_price_col = col
                    break
        
        print(f"\n저번 달 데이터 컬럼:")
        print(f"  - 종목코드: {prev_ticker_col}")
        print(f"  - 매수개수: {prev_shares_col}")
        if prev_price_col:
            print(f"  - 종가: {prev_price_col}")
        else:
            print(f"  - 종가: 찾을 수 없음 (저번달 종가 정보가 없습니다)")
        
        # 저번 달 포트폴리오를 딕셔너리로 변환 (종목코드: 매수개수)
        previous_dict = dict(zip(previous_portfolio[prev_ticker_col], previous_portfolio[prev_shares_col]))
        
        # 이번 달 포트폴리오를 딕셔너리로 변환 (종목코드: 매수개수)
        current_dict = dict(zip(current_portfolio['종목코드'], current_portfolio['매수개수']))
        
        # 1. 계속 보유해야 하는 종목 (이번 달에도 포함된 종목)
        common_tickers = set(previous_dict.keys()) & set(current_dict.keys())
        print(f"\n{'=' * 60}")
        print(f"계속 보유해야 하는 종목: {len(common_tickers)}개")
        print(f"{'=' * 60}")
        
        # 2. 주수 비교
        hold_unchanged = []  # 주수 변경 없음
        hold_adjusted = []   # 주수 조정 필요
        
        for ticker in common_tickers:
            prev_shares = previous_dict[ticker]
            curr_shares = current_dict[ticker]
            
            if prev_shares == curr_shares:
                hold_unchanged.append({
                    '종목코드': ticker,
                    '저번달_주수': int(prev_shares),
                    '이번달_주수': int(curr_shares),
                    '변화': '변화없음',
                    '이번달_종가': current_portfolio[current_portfolio['종목코드'] == ticker]['종가'].values[0],
                    '이번달_실제투자금액': current_portfolio[current_portfolio['종목코드'] == ticker]['실제투자금액'].values[0]
                })
            else:
                # 저번달 종가 가져오기
                prev_price = None
                if prev_price_col:
                    prev_price_row = previous_portfolio[previous_portfolio[prev_ticker_col] == ticker]
                    if len(prev_price_row) > 0:
                        prev_price = prev_price_row[prev_price_col].values[0]
                
                hold_adjusted.append({
                    '종목코드': ticker,
                    '저번달_주수': int(prev_shares),
                    '이번달_주수': int(curr_shares),
                    '주수변화': int(curr_shares - prev_shares),
                    '변화율(%)': round((curr_shares - prev_shares) / prev_shares * 100, 2) if prev_shares > 0 else 0,
                    '저번달_종가': prev_price if prev_price is not None else None,
                    '이번달_종가': current_portfolio[current_portfolio['종목코드'] == ticker]['종가'].values[0],
                    '이번달_실제투자금액': current_portfolio[current_portfolio['종목코드'] == ticker]['실제투자금액'].values[0]
                })
        
        # 3. 매도해야 하는 종목 (저번 달에만 있던 종목)
        sell_tickers = set(previous_dict.keys()) - set(current_dict.keys())
        
        # 4. 신규 매수 종목 (이번 달에만 있는 종목)
        buy_tickers = set(current_dict.keys()) - set(previous_dict.keys())
        
        # 매도종목: 현재종가·수익·수익률 계산 및 성과 요약 (len(sell_tickers) > 0 일 때)
        sell_df = None
        performance_summary = None
        if len(sell_tickers) > 0:
            # close_at_date 미정의 시 이번 달 기준일 종가 사용 (Step 3와 동일 로직)
            try:
                _close_at_date = close_at_date
            except NameError:
                close_date = target_date if target_date in close_df.index else close_df.index[close_df.index <= target_date].max()
                _close_at_date = close_df.loc[close_date] if close_date is not None else pd.Series(dtype=float)
            current_prices = _close_at_date.reindex(list(sell_tickers))
            sell_df = previous_portfolio[previous_portfolio[prev_ticker_col].isin(sell_tickers)].copy()
            sell_df['현재종가'] = sell_df[prev_ticker_col].map(current_prices).values
            if '실제투자금액' in sell_df.columns:
                sell_df['매수금액'] = sell_df['실제투자금액']
            elif prev_price_col is not None:
                sell_df['매수금액'] = sell_df[prev_price_col].astype(float) * sell_df[prev_shares_col].astype(int)
            else:
                sell_df['매수금액'] = np.nan
            sell_df['매도금액'] = sell_df['현재종가'] * sell_df[prev_shares_col].astype(int)
            sell_df['수익'] = sell_df['매도금액'] - sell_df['매수금액']
            sell_df['수익률(%)'] = (sell_df['수익'] / sell_df['매수금액'].replace(0, np.nan) * 100).round(2)
            # 성과 시트용: 현재종가 있는 행만 집계
            sell_df_valid = sell_df[sell_df['현재종가'].notna()]
            n_valid = len(sell_df_valid)
            n_no_price = len(sell_df) - n_valid
            if n_valid > 0:
                total_buy = sell_df_valid['매수금액'].sum()
                total_sell = sell_df_valid['매도금액'].sum()
                total_profit = total_sell - total_buy
                total_return_pct = round((total_profit / total_buy * 100), 2) if total_buy else None
                performance_summary = pd.DataFrame([{
                    '매도종목수': len(sell_tickers),
                    '현재종가_있는_종목수': n_valid,
                    '현재종가_없음_종목수': n_no_price,
                    '총_매수금액': total_buy,
                    '총_매도금액': total_sell,
                    '총_수익': total_profit,
                    '총_수익률(%)': total_return_pct
                }])
            else:
                performance_summary = pd.DataFrame([{
                    '매도종목수': len(sell_tickers),
                    '현재종가_있는_종목수': 0,
                    '현재종가_없음_종목수': n_no_price,
                    '총_매수금액': None,
                    '총_매도금액': None,
                    '총_수익': None,
                    '총_수익률(%)': None
                }])
        
        # 결과 출력
        print(f"\n[1] 주수 변경 없이 계속 보유: {len(hold_unchanged)}개")
        if len(hold_unchanged) > 0:
            hold_unchanged_df = pd.DataFrame(hold_unchanged)
            print(hold_unchanged_df.head(10).to_string(index=False))
            if len(hold_unchanged) > 10:
                print(f"... (총 {len(hold_unchanged)}개)")
        
        print(f"\n[2] 주수 조정 필요 (계속 보유): {len(hold_adjusted)}개")
        if len(hold_adjusted) > 0:
            hold_adjusted_df = pd.DataFrame(hold_adjusted)
            print(hold_adjusted_df.head(10).to_string(index=False))
            if len(hold_adjusted) > 10:
                print(f"... (총 {len(hold_adjusted)}개)")
            print(f"\n주수 조정 필요 종목 요약:")
            print(f"  - 주수 증가: {len([x for x in hold_adjusted if x['주수변화'] > 0])}개")
            print(f"  - 주수 감소: {len([x for x in hold_adjusted if x['주수변화'] < 0])}개")
        
        print(f"\n[3] 매도 필요: {len(sell_tickers)}개")
        if len(sell_tickers) > 0:
            sell_list = previous_portfolio[previous_portfolio[prev_ticker_col].isin(sell_tickers)][[prev_ticker_col, prev_shares_col]]
            print(sell_list.head(10).to_string(index=False))
            if len(sell_tickers) > 10:
                print(f"... (총 {len(sell_tickers)}개)")
        
        print(f"\n[4] 신규 매수: {len(buy_tickers)}개")
        if len(buy_tickers) > 0:
            buy_list = current_portfolio[current_portfolio['종목코드'].isin(buy_tickers)][['종목코드', '매수개수', '종가', '실제투자금액']]
            print(buy_list.head(10).to_string(index=False))
            if len(buy_tickers) > 10:
                print(f"... (총 {len(buy_tickers)}개)")
        
        # 결과를 DataFrame으로 정리
        comparison_summary = pd.DataFrame({
            '구분': ['계속보유(주수변화없음)', '계속보유(주수조정)', '매도', '신규매수'],
            '종목수': [len(hold_unchanged), len(hold_adjusted), len(sell_tickers), len(buy_tickers)]
        })
        
        print(f"\n{'=' * 60}")
        print("비교 요약")
        print(f"{'=' * 60}")
        print(comparison_summary.to_string(index=False))
        
        # 결과를 엑셀 파일에 추가 시트로 저장
        with pd.ExcelWriter(output_filename, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            if len(hold_unchanged) > 0:
                pd.DataFrame(hold_unchanged).to_excel(writer, sheet_name='계속보유_변화없음', index=False)
            if len(hold_adjusted) > 0:
                pd.DataFrame(hold_adjusted).to_excel(writer, sheet_name='계속보유_주수조정', index=False)
            if len(sell_tickers) > 0 and sell_df is not None:
                sell_df.to_excel(writer, sheet_name='매도종목', index=False)
            if performance_summary is not None:
                performance_summary.to_excel(writer, sheet_name='성과', index=False)
            if len(buy_tickers) > 0:
                buy_df = current_portfolio[current_portfolio['종목코드'].isin(buy_tickers)]
                buy_df.to_excel(writer, sheet_name='신규매수종목', index=False)
            comparison_summary.to_excel(writer, sheet_name='비교요약', index=False)
        
        print(f"\n✓ 비교 결과가 '{output_filename}' 파일의 추가 시트로 저장되었습니다.")
        print("\n" + "=" * 60)
        print("Step 4 완료!")
        print("=" * 60)
        
    except FileNotFoundError:
        print(f"경고: 파일을 찾을 수 없습니다: {PREVIOUS_QUARTER_FILE}")
        print("리밸런싱 비교를 건너뜁니다.")
    except Exception as e:
        print(f"경고: 리밸런싱 비교 중 오류 발생: {str(e)}")
        print("리밸런싱 비교를 건너뜁니다.")
        import traceback
        traceback.print_exc()
elif ENABLE_REBALANCING_COMPARISON and PREVIOUS_QUARTER_FILE is None:
    print("\n리밸런싱 비교 기능이 활성화되어 있지만 저번 분기 파일 경로가 설정되지 않았습니다.")
    print("PREVIOUS_QUARTER_FILE을 설정하면 비교 기능을 사용할 수 있습니다.")
else:
    print("\n리밸런싱 비교 기능이 비활성화되어 있습니다.")


리밸런싱 비교 기능이 비활성화되어 있습니다.


In [11]:
# ==================== 팩터 순위 조회 기능 ====================
print("\n" + "=" * 60)
print("팩터(profitability_v3) 순위 조회")
print("=" * 60)

# 기준일의 팩터 전체 값 (거래대금 필터 후 전체 종목 대상)
target_factor_all = factor_df.loc[target_date, liquid_tickers].dropna()

# 순위 계산 (팩터값 낮을수록 수익성 좋음 → 좋은 순위)
factor_sorted = target_factor_all.sort_values()
factor_ranked = pd.DataFrame({
    '종목코드': factor_sorted.index,
    '팩터값':   factor_sorted.values,
    '순위':     range(1, len(factor_sorted) + 1)
})
factor_ranked['백분위'] = (factor_ranked['순위'] / len(factor_ranked) * 100).round(2)

print(f"✓ 기준일: {target_date.strftime('%Y-%m-%d')}")
print(f"✓ 팩터 값이 있는 종목 수 (거래대금 필터 후): {len(factor_sorted):,}")

print(f"\n{'=' * 60}")
print("전체 유니버스 팩터 통계")
print(f"{'=' * 60}")
print(f"총 종목 수:  {len(factor_sorted):,}개")
print(f"최소 팩터값: {factor_sorted.min():.6f}  ← 수익성 최상위")
print(f"최대 팩터값: {factor_sorted.max():.6f}  ← 수익성 최하위")
print(f"평균 팩터값: {factor_sorted.mean():.6f}")
print(f"중앙값:      {factor_sorted.median():.6f}")
print(f"표준편차:    {factor_sorted.std():.6f}")

# 종목 조회
print(f"\n{'=' * 60}")
print("종목 조회 (종목코드를 입력하세요)")
print("여러 종목은 쉼표로 구분  |  종료: 'q'")
print(f"{'=' * 60}")

while True:
    user_input = input("\n종목코드 입력: ").strip()

    if user_input.lower() in ['q', 'quit', '종료']:
        print("조회를 종료합니다.")
        break

    if not user_input:
        print("종목코드를 입력해주세요.")
        continue

    query_tickers = [t.strip() for t in user_input.split(',')]

    for ticker in query_tickers:
        if ticker not in factor_ranked['종목코드'].values:
            print(f"  ⚠ '{ticker}' 종목을 찾을 수 없습니다.")
            continue

        info       = factor_ranked[factor_ranked['종목코드'] == ticker].iloc[0]
        rank       = int(info['순위'])
        factor_val = info['팩터값']
        pct        = info['백분위']

        print(f"\n  종목코드: {ticker}")
        print(f"  팩터값:   {factor_val:.6f}")
        print(f"  순위:     {rank:,}위 / {len(factor_ranked):,}개  (낮을수록 수익성 좋음)")
        print(f"  백분위:   상위 {pct:.2f}%")

        if rank <= 10:
            print(f"  → 수익성 최상위 {rank}위!")
        elif pct <= 20:
            print(f"  → 수익성 상위 20% 이내 (팩터 1분위 선정 대상)")
        elif pct <= 40:
            print(f"  → 수익성 상위 20~40% (팩터 2분위)")
        else:
            print(f"  → 팩터 {int(pct//20)+1}분위")

    print(f"\n{'=' * 60}")


팩터(profitability_v3) 순위 조회


KeyError: "['알테오젠', '삼성에피스홀딩스', '현대오토에버', '이수페타시스', '리가켐바이오', 'HLB', '한미약품', '이오테크닉스', '엔씨소프트', '산일전기', '에임드바이오', '대한조선', '우리기술', '영원무역홀딩스', '이수스페셜티케미컬', '서진시스템', '한미사이언스', 'F&F', '한국카본', '태성', '알지노믹스', '실리콘투', '리브스메드', '코오롱인더', '씨어스테크놀로지', '시프트업', '달바글로벌', '피에스케이', '차바이오텍', '파라다이스', '아이티센글로벌', '휴림로봇', '로킷헬스케어', 'STX엔진', '씨엠티엑스', '태광산업', '에이치브이엠', '엔켐', '제이에스링크', '인벤티지랩', '두산테스나', '네이처셀', 'NHN', 'SFA반도체', '씨아이에스', '제이앤티씨', '필옵틱스', '루닛', '에스에프에이', '한샘', '현대ADM', '세진중공업', '기가비스', '휴온스글로벌', '아이에스동서', '해성디에스', '케이씨텍', '원익QnC', '명인제약', '세방전지', '프로티나', '세미파이브', '노타', '후성', '코오롱', '동원시스템즈', 'F&F홀딩스', '티에프이', '동아쏘시오홀딩스', '대한해운', '한전산업', '한국쉘석유', '큐로셀', '세아홀딩스', '에이디테크놀로지', '하나투어', '삼양바이오팜', '인카금융서비스', '큐렉소', '한스바이오메드', '일진하이솔루스', '아세아', '비에이치', '이뮨온시아', '프레스티지바이오파마', 'GRT', '코스모화학', '고려제강', '덴티움', '삼양홀딩스', '삼양컴텍', '바이오노트', '신대양제지', '한일홀딩스', '제우스', '태영건설', '명신산업', '비츠로넥스텍', '삼화콘덴서', 'NICE', '한세실업', 'HEM파마', '에코마케팅', '나라스페이스테크놀로지', 'HLB제약', '동화기업', '에스바이오메딕스', '그린광학', '미코', 'DI동일', 'HLB생명과학', '퍼